![Fraud detection image](cover_image.jpg)

🏦 Banks are battling frauds with machine learning models, but changing data patterns can weaken these defenses. London's Poundbank needs your help to figure out why their fraud detection models aren't as accurate anymore.

Poundbank recommends the `nannyml` library for monitoring machine learning models, which is also their tool of choice.

## The data

They have provided you with a reference(test data) and analysis set(production data). A summary and preview are provided below.

## reference.csv and analysis.csv

| Column     | Description              |
|------------|--------------------------|
| `'timestamp'` | Date of the transaction. |
| `'time_since_login_min'` | Time since the user logged in to the app. |
| `'transaction_amount'` | The amount of Pounds(£) that users sent to another account. |
| `'transaction_type'` | Transaction type: <ul><li>`CASH-OUT` - Withdrawing money from an account.</li><li>`PAYMENT` - Transaction where a payment is made to a third party.</li><li>`CASH-IN` - This is the opposite of a cash-out. It involves depositing money into an account.</li><li>`TRANSFER` - Transaction which involves moving funds from one account to another.</li> |
| `'is_first_transaction'` | A binary indicator denoting if the transaction is the user's first (1 for the first transaction, 0 otherwise). |
| `'user_tenure_months'` | The duration in months since the user's account was created or since they became a member. |
| `'is_fraud'` | A binary label indicating whether the transaction is fraudulent (1 for fraud, 0 otherwise). |
| `'predicted_fraud_proba'` | The probability assigned by a detection model indicates the likelihood of a fraudulent transaction. |
| `'predicted_fraud'` |  The predicted classification label is calculated based on predicted fraud probability by the detection model (1 for predicted fraud, 0 otherwise). |

In [41]:
!pip install nannyml

Defaulting to user installation because normal site-packages is not writeable


In [50]:
# Import required libraries
import pandas as pd
import nannyml as nml
nml.disable_usage_logging()

reference = pd.read_csv("reference.csv")
analysis = pd.read_csv("analysis.csv")
reference.head()

,timestamp,time_since_login_min,transaction_amount,transaction_type,is_first_transaction,user_tenure_months,is_fraud,predicted_fraud_proba,predicted_fraud
0,2018-01-01 00:00:00.000,1.561750,3981.1,PAYMENT,False,0.318980,1.0,0.99,1
1,2018-01-01 00:08:43.152,1.658074,1267.9,PAYMENT,False,7.391323,0.0,0.07,0
2,2018-01-01 00:17:26.304,2.454287,1984.7,CASH-IN,False,0.781225,1.0,1.00,1
3,2018-01-01 00:26:09.456,2.392085,2265.2,CASH-OUT,False,0.680473,1.0,0.98,1
4,2018-01-01 00:34:52.608,2.189806,2126.8,CASH-IN,False,8.542895,1.0,0.99,1


In [43]:
estimator = nml.CBPE(
    y_pred_proba='predicted_fraud_proba',
    y_pred='predicted_fraud',
    y_true='is_fraud',
    timestamp_column_name='timestamp',
    metrics=['accuracy'],
    chunk_period='M',
    problem_type='classification_binary'
)

estimator.fit(reference)
estimated_results = estimator.estimate(analysis)

estimated_df = estimated_results.to_df()
estimated_df.columns = ['_'.join([c for c in col if c]).strip('_') 
                         for col in estimated_df.columns]

print("Estimated Columns:", estimated_df.columns.tolist())
print("\nFull Estimated Results:")
print(estimated_df.to_string())

Estimated Columns: ['chunk_key', 'chunk_chunk_index', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date', 'chunk_period', 'accuracy_value', 'accuracy_sampling_error', 'accuracy_realized', 'accuracy_upper_confidence_boundary', 'accuracy_lower_confidence_boundary', 'accuracy_upper_threshold', 'accuracy_lower_threshold', 'accuracy_alert']

Full Estimated Results:
   chunk_key  chunk_chunk_index  chunk_start_index  chunk_end_index chunk_start_date                chunk_end_date chunk_period  accuracy_value  accuracy_sampling_error  accuracy_realized  accuracy_upper_confidence_boundary  accuracy_lower_confidence_boundary  accuracy_upper_threshold  accuracy_lower_threshold  accuracy_alert
0    2018-01                  0                  0             5119       2018-01-01 2018-01-31 23:59:59.999999999    reference        0.944593                 0.003224           0.947070                            0.954264                            0.934922                  0.9508

In [44]:
calc = nml.PerformanceCalculator(
    y_pred_proba='predicted_fraud_proba',
    y_pred='predicted_fraud',
    y_true='is_fraud',
    timestamp_column_name='timestamp',
    metrics=['accuracy'],
    chunk_period='M',
    problem_type='classification_binary'
)

calc.fit(reference)
realized_results = calc.calculate(analysis)

realized_df = realized_results.to_df()
realized_df.columns = ['_'.join([c for c in col if c]).strip('_') 
                        for col in realized_df.columns]

print("Realized Columns:", realized_df.columns.tolist())
print("\nFull Realized Results:")
print(realized_df.to_string())

Realized Columns: ['chunk_key', 'chunk_chunk_index', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date', 'chunk_period', 'chunk_targets_missing_rate', 'accuracy_sampling_error', 'accuracy_value', 'accuracy_upper_threshold', 'accuracy_lower_threshold', 'accuracy_alert']

Full Realized Results:
   chunk_key  chunk_chunk_index  chunk_start_index  chunk_end_index chunk_start_date                chunk_end_date chunk_period  chunk_targets_missing_rate  accuracy_sampling_error  accuracy_value  accuracy_upper_threshold  accuracy_lower_threshold  accuracy_alert
0    2018-01                  0                  0             5119       2018-01-01 2018-01-31 23:59:59.999999999    reference                         0.0                 0.003224        0.947070                  0.950808                  0.936367           False
1    2018-02                  1               5120             9744       2018-02-01 2018-02-28 23:59:59.999999999    reference                       

In [45]:
# Print ALL columns so we can identify exact names
print("ESTIMATED columns:", estimated_df.columns.tolist())
print("REALIZED columns:", realized_df.columns.tolist())

# Show all alert-related columns
est_alert_cols = [c for c in estimated_df.columns if 'alert' in c]
rea_alert_cols = [c for c in realized_df.columns if 'alert' in c]
print("\nEstimated alert cols:", est_alert_cols)
print("Realized alert cols:", rea_alert_cols)

# Show key columns
est_key_cols = [c for c in estimated_df.columns if 'key' in c or 'start' in c or 'end' in c]
rea_key_cols = [c for c in realized_df.columns if 'key' in c or 'start' in c or 'end' in c]
print("\nEstimated key cols:", est_key_cols)
print("Realized key cols:", rea_key_cols)

# Show all rows with their alert status
est_alert_col = est_alert_cols[0]
rea_alert_col = rea_alert_cols[0]
est_key_col   = est_key_cols[0]
rea_key_col   = rea_key_cols[0]

print("\n--- All Estimated rows ---")
print(estimated_df[[est_key_col, est_alert_col]].to_string())

print("\n--- All Realized rows ---")
print(realized_df[[rea_key_col, rea_alert_col]].to_string())

# Collect months where BOTH estimated AND realized trigger alerts
est_alerts = set(estimated_df[estimated_df[est_alert_col] == True][est_key_col].tolist())
rea_alerts = set(realized_df[realized_df[rea_alert_col] == True][rea_key_col].tolist())

print("\nEstimated alert keys:", est_alerts)
print("Realized alert keys:", rea_alerts)

# Intersection - months where BOTH triggered
both_alerts = est_alerts.intersection(rea_alerts)
print("Months with BOTH alerts:", both_alerts)

# Convert to month_year format
def key_to_month_year(key):
    key = str(key).strip().split(' ')[0].strip('[').strip('(')
    try:
        dt = datetime.strptime(key[:7], "%Y-%m")
        return dt.strftime("%B_%Y").lower()
    except:
        return key

months_with_performance_alerts = sorted([key_to_month_year(k) for k in both_alerts])
print("\nmonths_with_performance_alerts =", months_with_performance_alerts)

ESTIMATED columns: ['chunk_key', 'chunk_chunk_index', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date', 'chunk_period', 'accuracy_value', 'accuracy_sampling_error', 'accuracy_realized', 'accuracy_upper_confidence_boundary', 'accuracy_lower_confidence_boundary', 'accuracy_upper_threshold', 'accuracy_lower_threshold', 'accuracy_alert']
REALIZED columns: ['chunk_key', 'chunk_chunk_index', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date', 'chunk_period', 'chunk_targets_missing_rate', 'accuracy_sampling_error', 'accuracy_value', 'accuracy_upper_threshold', 'accuracy_lower_threshold', 'accuracy_alert']

Estimated alert cols: ['accuracy_alert']
Realized alert cols: ['accuracy_alert']

Estimated key cols: ['chunk_key', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date']
Realized key cols: ['chunk_key', 'chunk_start_index', 'chunk_end_index', 'chunk_start_date', 'chunk_end_date']

--- All Estimated rows ---
   chu

In [46]:
feature_cols     = ['time_since_login_min', 'transaction_amount', 
                    'transaction_type', 'is_first_transaction', 'user_tenure_months']
numerical_cols   = ['time_since_login_min', 'transaction_amount', 'user_tenure_months']
categorical_cols = ['transaction_type', 'is_first_transaction']

univariate_calc = nml.UnivariateDriftCalculator(
    column_names=feature_cols,
    timestamp_column_name='timestamp',
    chunk_period='M',
    continuous_methods=['kolmogorov_smirnov'],
    categorical_methods=['chi2']
)

univariate_calc.fit(reference)
drift_results = univariate_calc.calculate(analysis)

drift_df = drift_results.to_df()

# Flatten MultiIndex columns
drift_df.columns = ['_'.join([c for c in col if c]).strip('_') 
                     for col in drift_df.columns]

print("Drift columns:", drift_df.columns.tolist())

# Count alerts per feature
alert_counts = {}
for col in numerical_cols:
    alert_col = f'{col}_kolmogorov_smirnov_alert'
    if alert_col in drift_df.columns:
        alert_counts[col] = drift_df[alert_col].sum()
        print(f"{col} KS alerts: {alert_counts[col]}")

for col in categorical_cols:
    alert_col = f'{col}_chi2_alert'
    if alert_col in drift_df.columns:
        alert_counts[col] = drift_df[alert_col].sum()
        print(f"{col} Chi2 alerts: {alert_counts[col]}")

highest_correlation_feature = max(alert_counts, key=alert_counts.get)
print("\nhighest_correlation_feature =", highest_correlation_feature)

Drift columns: ['chunk_chunk_key', 'chunk_chunk_chunk_index', 'chunk_chunk_start_index', 'chunk_chunk_end_index', 'chunk_chunk_start_date', 'chunk_chunk_end_date', 'chunk_chunk_period', 'time_since_login_min_kolmogorov_smirnov_value', 'time_since_login_min_kolmogorov_smirnov_upper_threshold', 'time_since_login_min_kolmogorov_smirnov_lower_threshold', 'time_since_login_min_kolmogorov_smirnov_alert', 'transaction_amount_kolmogorov_smirnov_value', 'transaction_amount_kolmogorov_smirnov_upper_threshold', 'transaction_amount_kolmogorov_smirnov_lower_threshold', 'transaction_amount_kolmogorov_smirnov_alert', 'user_tenure_months_kolmogorov_smirnov_value', 'user_tenure_months_kolmogorov_smirnov_upper_threshold', 'user_tenure_months_kolmogorov_smirnov_lower_threshold', 'user_tenure_months_kolmogorov_smirnov_alert', 'is_first_transaction_chi2_value', 'is_first_transaction_chi2_upper_threshold', 'is_first_transaction_chi2_lower_threshold', 'is_first_transaction_chi2_alert', 'transaction_type_chi2

In [47]:
analysis_copy = analysis.copy()
analysis_copy['timestamp'] = pd.to_datetime(analysis_copy['timestamp'])

# Monthly average transaction amount in analysis
monthly_avg = (
    analysis_copy
    .groupby(analysis_copy['timestamp'].dt.to_period('M'))['transaction_amount']
    .mean()
    .reset_index()
)
monthly_avg.columns = ['month', 'avg_transaction_amount']

ref_avg = reference['transaction_amount'].mean()
ref_std  = reference['transaction_amount'].std()

print(f"Reference avg: {ref_avg:.4f}")
print(f"Reference std: {ref_std:.4f}")
print("\nAll monthly averages:")
print(monthly_avg.to_string())

# Find the month with maximum deviation from reference mean
monthly_avg['deviation'] = abs(monthly_avg['avg_transaction_amount'] - ref_avg)
most_deviated = monthly_avg.loc[monthly_avg['deviation'].idxmax()]

print("\nMost deviated month:")
print(most_deviated)

alert_avg_transaction_amount = round(float(most_deviated['avg_transaction_amount']), 1)
print(f"\nalert_avg_transaction_amount = {alert_avg_transaction_amount}")

Reference avg: 2964.7444
Reference std: 2039.3410

All monthly averages:
     month  avg_transaction_amount
0  2018-11             2994.880061
1  2018-12             2979.182770
2  2019-01             2993.979414
3  2019-02             3000.146194
4  2019-03             2968.669785
5  2019-04             2923.636529
6  2019-05             2996.924907
7  2019-06             3069.818355

Most deviated month:
month                         2019-06
avg_transaction_amount    3069.818355
deviation                  105.074001
Name: 7, dtype: object

alert_avg_transaction_amount = 3069.8


In [48]:
print("=" * 60)
print("FINAL ANSWERS")
print("=" * 60)
print(f"months_with_performance_alerts = {months_with_performance_alerts}")
print(f"highest_correlation_feature    = '{highest_correlation_feature}'")
print(f"alert_avg_transaction_amount   = {alert_avg_transaction_amount}")

FINAL ANSWERS
months_with_performance_alerts = ['april_2019', 'june_2019', 'may_2019']
highest_correlation_feature    = 'time_since_login_min'
alert_avg_transaction_amount   = 3069.8
